# TirraMind Phase 50 — Eval Only Notebook

This notebook is intentionally separate from training.

- No retraining
- Load trained artifacts (`epoch_090.pt` + `gnn_model_phase50.pt`)
- Run walk-forward backtest/evaluation only


In [ ]:
import json
from pathlib import Path

# Aligned canonical 21d forward-return labels (DATA_FIX, 2026-06-08).
# Full 40-fold IC on Kaggle CPU (~30–60 min). Set eval_smoke=True for ~10 min smoke.
EVAL_CONFIG = {
    "phase": "50-eval-aligned",
    "source_version": 52,
    "target_epoch": 90,
    "require_artifact_min_bytes": 1_000_000,
    "eval_smoke": False,
    "primary_ic_strategy": "GNN-ConcatReturnHead",
    "skip_purged_ranker": True,
    "run_honest_baseline": True,
    "run_data_label_audit": True,
}

print(json.dumps(EVAL_CONFIG, indent=2))


In [ ]:
import subprocess
import sys

def pip(*args):
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", *args], check=True)

pip("torch-geometric==2.7.0")
pip("rich", "tqdm", "scipy>=1.14.0")

import torch
import torch_geometric
import scipy
print("torch", torch.__version__)
print("pyg", torch_geometric.__version__)
print("scipy", scipy.__version__)


In [ ]:
import os
import shutil
import sys
from pathlib import Path

WORK_DIR = Path("/kaggle/working/tirramind_v1")
WORK_DIR.mkdir(parents=True, exist_ok=True)

print("Mounted input datasets:")
for p in sorted(Path("/kaggle/input").iterdir()):
    print(" ", p.name)

def find_code_root(root="/kaggle/input"):
    for dirpath, dirs, _ in os.walk(root):
        if {"agent", "scripts"}.issubset(set(dirs)):
            return Path(dirpath)
    return None

def find_data_root(root="/kaggle/input"):
    for dirpath, _, files in os.walk(root):
        if "pipeline.db" in files:
            return Path(dirpath)
    return None

_code_root = find_code_root()
if _code_root is None:
    raise RuntimeError(
        "CODE NOT FOUND under /kaggle/input. Attach deeperisbetter/tirramind-code."
    )

for name in ("agent", "scripts"):
    dst = WORK_DIR / name
    if dst.exists():
        shutil.rmtree(dst)
    shutil.copytree(_code_root / name, dst)

_data_root = find_data_root()
if _data_root is None:
    raise RuntimeError(
        "DATA NOT FOUND under /kaggle/input. Attach deeperisbetter/tirramind-data."
    )

PIPELINE_DIR = WORK_DIR / ".tirra_pipeline"
PIPELINE_DIR.mkdir(parents=True, exist_ok=True)
PIPELINE_DB_DST = PIPELINE_DIR / "pipeline.db"
shutil.copy2(_data_root / "pipeline.db", PIPELINE_DB_DST)

sys.path.insert(0, str(WORK_DIR))
print(f"Code from {_code_root}")
print(f"Data from {_data_root}")
print(f"pipeline.db -> {PIPELINE_DB_DST}")


In [ ]:
from pathlib import Path

def find_best_file(filename: str, min_bytes: int) -> Path:
    candidates = []
    for p in Path("/kaggle/input").rglob(filename):
        try:
            size = p.stat().st_size
        except OSError:
            continue
        if size >= min_bytes:
            candidates.append((size, p))
    if not candidates:
        raise FileNotFoundError(f"No valid {filename} >= {min_bytes} bytes under /kaggle/input")
    candidates.sort(key=lambda x: x[0], reverse=True)
    return candidates[0][1]

model_src = find_best_file("gnn_model_phase50.pt", EVAL_CONFIG["require_artifact_min_bytes"])
epoch_src = find_best_file(f"epoch_{EVAL_CONFIG['target_epoch']:03d}.pt", EVAL_CONFIG["require_artifact_min_bytes"])

print("model:", model_src, model_src.stat().st_size)
print("epoch:", epoch_src, epoch_src.stat().st_size)


In [ ]:
import subprocess
import sys
from pathlib import Path

out_json = Path("/kaggle/working/ic_results_eval_phase50.json")

cmd = [
    sys.executable,
    "scripts/phase40_gnn_backtest.py",
    "--model-path", str(model_src),
    "--weights-from-epoch", str(epoch_src),
    "--db-path", ".tirra_pipeline/pipeline.db",
    "--out", str(out_json),
    "--primary-ic-strategy", EVAL_CONFIG["primary_ic_strategy"],
]
if EVAL_CONFIG.get("skip_purged_ranker", True):
    cmd.append("--skip-purged-ranker")
if EVAL_CONFIG.get("eval_smoke", False):
    cmd.append("--smoke")

print("Running:", " ".join(cmd))
proc = subprocess.Popen(cmd, cwd=str(WORK_DIR), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
assert proc.stdout is not None
for line in proc.stdout:
    print(line, end="")
rc = proc.wait()
if rc != 0:
    raise RuntimeError(f"phase40_gnn_backtest failed with exit code {rc}")
print("Done. Results:", out_json)


In [ ]:
import subprocess
import sys
from pathlib import Path

if not EVAL_CONFIG.get("run_honest_baseline", True):
    print("Skipping honest baseline audit")
else:
    out = Path("/kaggle/working/honest_baseline_audit_full.json")
    cmd = [
        sys.executable,
        "scripts/honest_baseline_audit.py",
        "--out", str(out),
        "--checkpoint", str(model_src),
        "--weights-from-epoch", str(epoch_src),
    ]
    if EVAL_CONFIG.get("eval_smoke", False):
        cmd.append("--smoke")
    print("Running:", " ".join(cmd))
    subprocess.run(cmd, cwd=str(WORK_DIR), check=True)
    print("Done:", out)

In [ ]:
import subprocess
import sys
from pathlib import Path

if not EVAL_CONFIG.get("run_data_label_audit", True):
    print("Skipping data label audit")
else:
    out = Path("/kaggle/working/data_label_audit_full.json")
    cmd = [
        sys.executable,
        "scripts/data_label_audit.py",
        "--out", str(out),
    ]
    if EVAL_CONFIG.get("eval_smoke", False):
        cmd.append("--smoke")
    print("Running:", " ".join(cmd))
    subprocess.run(cmd, cwd=str(WORK_DIR), check=True)
    print("Done:", out)

In [ ]:
import json
from pathlib import Path

def _load_json(path: Path) -> dict:
    if not path.exists():
        return {}
    return json.loads(path.read_text())

ic = _load_json(Path("/kaggle/working/ic_results_eval_phase50.json"))
baseline = _load_json(Path("/kaggle/working/honest_baseline_audit_full.json"))
audit = _load_json(Path("/kaggle/working/data_label_audit_full.json"))

print("=" * 60)
print("ALIGNED EVAL SUMMARY (canonical 21d forward returns)")
print("=" * 60)

gate = ic.get("primary_gate", {})
print(f"\nGNN primary ({gate.get('strategy', '?')}): "
      f"IC={gate.get('mean_ic', 0):+.4f}  t={gate.get('t_stat', 0):+.2f}  "
      f"n={gate.get('n_folds', 0)}  PASS={gate.get('passed', False)}")

for name, m in ic.get("strategies", {}).items():
    print(f"  {name:22s} IC={m.get('mean_ic', 0):+.4f}  t={m.get('t_stat', 0):+.2f}  n={m.get('n_folds', 0)}")

if baseline:
    print("\nHonest baselines:")
    for name, m in baseline.get("ic_results", {}).items():
        print(f"  {name:22s} IC={m.get('mean_ic', 0):+.4f}  t={m.get('t_stat', 0):+.2f}  n={m.get('n_folds', 0)}")
    print("  recommendation:", baseline.get("recommendation", "?"))

if audit:
    print("\nData label audit verdict:", audit.get("verdict", "?"))
